# microWakeWord — Train any wake word
This is an updated version of microWakeWord_train_any_wakeword by [Alfie Dennen](https://github.com/alfiedennen/microwakeword-trainer) that just pins to specific versions and downloads the file rather than access google drive

A single-notebook trainer for custom wake words on ESPHome `micro_wake_word` devices
(M5Stack Atom Echo, Voice PE, etc).

Independent training-data sources (configured in the first code cell):
- **Positive** and **negative** sources: `generated`, `uploaded`, or `both`.
- **Background** source: `standard` (FMA), `uploaded`, or `both`.

ZIP archives may contain nested WAV, FLAC, MP3, OGG or M4A files. One Colab upload
prompt accepts all required ZIPs, named `positive*.zip`, `negative*.zip` and/or
`background*.zip` (multiple archives per category supported). All uploaded audio
is converted to 16 kHz mono PCM WAV. Upload prompts are skipped when none are needed.

## Required runtime
**Runtime → Change runtime type → A100 GPU + High-RAM**.
T4 OOMs during validation. A100 + High-RAM gives 40 GB VRAM + 85 GB system RAM.

## What you get
A `<output_name>.tflite` (~60 KB) + companion `.json` manifest, ready to drop into
your ESPHome config under `micro_wake_word: models:`.

## Workflow
1. Edit the **CONFIGURE HERE** cell below for your wake word
2. **Runtime → Run all** (complete the ZIP prompts in upload-enabled modes)
3. Find `<output_name>.tflite` + `.json` in the final output folder / browser download when it's done
4. Test on hardware — likely needs manifest tuning (cutoff, sliding_window) for
   your specific model. See the deployment notes at the end.

Built from working production deployment of "Hey Harold" — all known
upstream bugs are patched in this notebook.

### Optional custom background audio
Set `BACKGROUND_SOURCE` independently to `"standard"` (default), `"uploaded"`, or
`"both"`. Background clips are augmentation material, **not** labelled negatives.

### Independent data sources and Piper languages

The first code cell now controls `POSITIVE_SOURCE`, `NEGATIVE_SOURCE` (each: `generated`, `uploaded`, `both`) and `BACKGROUND_SOURCE` (`standard`, `uploaded`, `both`). Upload all required ZIPs in a single Colab prompt, with filenames starting `positive`, `negative`, or `background`. You can supply multiple ZIPs per category. **The random audio previews use generated positive and generated confusable-negative samples only; they never play uploaded audio.**

Piper defaults to the original multi-speaker English `.pt` generator. For other languages choose `PIPER_MODEL_TYPE = "voice"`, select `PIPER_LANGUAGE`, `PIPER_VOICE` and `PIPER_QUALITY`, and provide a suitable wake-word IPA. The voice must exist in the Piper voices repository, or you must provide direct `.onnx` and `.onnx.json` URLs. Ordinary Piper `.onnx` voices may be slower and have less speaker variety than the English generator. Phonemes absent from a selected voice's inventory cannot be synthesised accurately.


In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURE YOUR WAKE WORD HERE                       ║
# ╚═══════════════════════════════════════════════════════════════════════╝

# ─── Wake word identity ───
WAKE_WORD = "Hey Harold"           # Human-readable name (shown in HA)
OUTPUT_NAME = "hey_harold"         # Filename (no spaces, lowercase)
AUTHOR = "your_name"               # Goes in the manifest
AUTHOR_WEBSITE = "https://github.com/yourname"

# ─── Independent training data sources ───
# "generated": original pipeline (Piper positives / standard negatives), no ZIP prompt
# "uploaded": only the uploaded dataset, ZIP upload required
# "both": generated/standard data combined with the uploaded dataset
# Positive and negative sources can now be configured independently.
POSITIVE_SOURCE = "generated"  # "generated", "uploaded", or "both"
NEGATIVE_SOURCE = "generated"  # "generated", "uploaded", or "both"
assert POSITIVE_SOURCE in {"generated", "uploaded", "both"}, "Invalid POSITIVE_SOURCE"
assert NEGATIVE_SOURCE in {"generated", "uploaded", "both"}, "Invalid NEGATIVE_SOURCE"

# ─── Background audio used for augmentation (independent of other sources) ───
# "standard": use the original FMA background dataset; no upload prompt
# "uploaded": prompt for background-audio ZIP instead of downloading FMA
# "both": combine original FMA audio and your uploaded background ZIP
BACKGROUND_SOURCE = "standard"  # "standard", "uploaded", or "both"
assert BACKGROUND_SOURCE in {"standard", "uploaded", "both"}, "Invalid BACKGROUND_SOURCE"

# ─── Piper voice and language selection (used only for generated audio) ───
# "generator": existing high-diversity English LibriTTS-R checkpoint (.pt).
# "voice": ordinary Piper ONNX voice for your chosen language/locale.
# The phonetic IPA input is preserved for generated POSITIVE examples in both modes.
# Find voice IDs at https://huggingface.co/rhasspy/piper-voices/tree/main
PIPER_MODEL_TYPE = "generator"  # "generator" or "voice"
PIPER_LANGUAGE = "en_US"         # e.g. "en_US", "en_GB", "fr_FR", "de_DE"
PIPER_VOICE = "lessac"            # Piper voice directory/name, e.g. "lessac"
PIPER_QUALITY = "medium"         # Piper voice quality directory, e.g. "medium"
# A custom direct ONNX URL is optional (must also supply its matching .onnx.json URL).
PIPER_VOICE_URL = None
PIPER_VOICE_CONFIG_URL = None
assert PIPER_MODEL_TYPE in {"generator", "voice"}, "Invalid PIPER_MODEL_TYPE"
if PIPER_MODEL_TYPE == "generator" and PIPER_LANGUAGE != "en_US":
    raise ValueError("The bundled LibriTTS-R generator is en_US only. Select PIPER_MODEL_TYPE='voice' for other languages.")

# ─── If POSITIVE_SOURCE includes generated samples ───
# Look up IPA for your wake word: https://www.internationalphoneticassociation.org/IPA-chart
# or use eSpeak: `espeak-ng -q --ipa "Hey Harold"` on Linux
WAKE_WORD_IPA_US = "hˈeɪ hˈærəld"   # US English pronunciation
WAKE_WORD_IPA_UK = "hˈeɪ hˈɛrəld"   # UK English (set to None to skip second pass)
SAMPLES_US = 30000                  # ~12 min on T4, ~7 min on A100
SAMPLES_UK = 15000                  # 0 to skip UK pass

# Confusable phrases — should NOT trigger your wake word.
# Include: phonetic neighbors, the bare name without prefix, common
# false-trigger phrases like other assistant names.
CONFUSABLE_PHRASES = [
    # General "hey X" near-misses
    "hey there", "hey you", "hey y'all", "hey now",
    # Other assistant wake words (must not steal yours)
    "hey siri", "hey google", "okay google", "hey alexa", "okay nabu",
    # WAKE-WORD SPECIFIC — replace these for your word!
    "hey howard", "hey harvey", "hey gerald", "hey carol",  # H-name neighbors
    "harold", "the herald",                                  # bare name + similar
    "hairy old", "hello harold",                              # phonetic mash-ups
]
SAMPLES_PER_CONFUSABLE = 1000   # 500 = light, 1000 = recommended, 2000 = max

# ─── Manifest tuning (works well as defaults; tune after on-device testing) ───
PROBABILITY_CUTOFF = 0.85          # 0.5 = too lenient, 0.97 = okay_nabu strict
SLIDING_WINDOW_SIZE = 5            # 5 = okay_nabu default; higher = slower fire
TENSOR_ARENA_SIZE = 50000          # 30000 works, 50000 has more headroom

# ─── Languages this model is trained for (manifest metadata) ───
TRAINED_LANGUAGES = ["en"]         # add "it", "es" etc if you have multi-accent samples

# ╔═══════════════════════════════════════════════════════════════════════╗
# ║                           END OF CONFIG                                ║
# ╚═══════════════════════════════════════════════════════════════════════╝
print(f"Training {WAKE_WORD!r} as {OUTPUT_NAME}: positive={POSITIVE_SOURCE}, negative={NEGATIVE_SOURCE}, background={BACKGROUND_SOURCE}, piper={PIPER_MODEL_TYPE}/{PIPER_LANGUAGE}")


In [ ]:
import shutil
from pathlib import Path

RESET_DIRS = [
    "/content/generated_samples",
    "/content/confusable_negatives",
    "/content/generated_augmented_features",
    "/content/confusable_features",
    "/content/real_recording_features",
    "/content/uploaded_positive_samples",
    "/content/uploaded_negative_samples",
    "/content/uploaded_positive_features",
    "/content/uploaded_negative_features",
    "/content/uploaded_background_audio",
]

for path in RESET_DIRS:
    p = Path(path)
    if p.exists():
        print(f"Removing {p}")
        shutil.rmtree(p)

print("Wake-word-specific training data cleared.")

In [ ]:
# === Local output only — no Google Drive ===
import os

OUTPUT_DIR = "/content/final_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output folder: {OUTPUT_DIR}")

In [ ]:
# === Install microWakeWord (kernel-restart-free) ===
# Workarounds for two upstream bugs:
#  1. kahrendt/microWakeWord setup.py has no find_packages() — non-editable
#     install skips the audio/ subpackage. Editable install needs kernel
#     restart, breaks Run All. Fix: install deps + sys.path.insert().
#  2. train.py calls .numpy() on values that newer TF returns as numpy
#     arrays already. Patch with hasattr() guard.
import os, sys, subprocess, importlib, re

DEPS = [
    'audiomentations==0.43.1', 'datasets==5.1.0', 'mmap_ninja==0.9.0', 'numpy==2.5.3',
    'pymicro-features==2.0.2', 'pyyaml==6.0.3', 'tensorflow==2.21.0', 'webrtcvad-wheels==2.0.14.post1',
    'ai-edge-litert==2.2.0',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
MWW_COMMIT = "4665173"

print('Installing dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS, check=True)

if not os.path.exists('microWakeWord'):
    subprocess.run([
        "git", "clone", "https://github.com/kahrendt/microWakeWord", "microWakeWord" ], check=True)

subprocess.run(["git", "-C", "microWakeWord", "checkout", MWW_COMMIT], check=True)

MWW_DIR = '/content/microWakeWord'
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()

fp = '/content/microWakeWord/microwakeword/train.py'
src = open(fp).read()
patched = re.sub(
    r'(\b[a-zA-Z_]+\["[a-z]+"\])\.numpy\(\)',
    r'(\1.numpy() if hasattr(\1, "numpy") else \1)',
    src
)
n = patched.count('hasattr') - src.count('hasattr')
if n > 0:
    open(fp, 'w').write(patched)
    print(f'Patched {n} .numpy() calls in train.py')

import microwakeword
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print('OK: microwakeword.audio.* imports clean')


In [ ]:
# === Generated training data (optional) ===
if POSITIVE_SOURCE in ("generated", "both") or NEGATIVE_SOURCE in ("generated", "both"):
    # === Piper sample generator install ===
    import glob, os, shutil, subprocess, sys, urllib.request
    os.chdir('/content')
    print('Piper sample-gen cells will produce samples')

    PIPER_REPO_DIR = '/content/piper'
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    PIPER_COMMIT = "73c04d8"
    PIPER_SAMPLE_GENERATOR_COMMIT = "2971426a55072f7d22fec416ca7800df8bd23207"

    subprocess.run(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'pip==26.2.1', 'setuptools==84.0.0', 'wheel==0.48.0', 'cython==3.3.0'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                    'piper-tts==1.8.0', 'piper-sample-generator==3.2.0'], check=True)

    if not os.path.exists(PIPER_REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/rhasspy/piper', PIPER_REPO_DIR], check=True)
    subprocess.run([ "git", "-C", PIPER_REPO_DIR, "checkout", "--detach", PIPER_COMMIT], check=True)

    if not os.path.exists(PIPER_SAMPLE_GENERATOR_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/rhasspy/piper-sample-generator',
                    PIPER_SAMPLE_GENERATOR_DIR], check=True)
    subprocess.run([ "git", "-C", PIPER_SAMPLE_GENERATOR_DIR, "checkout", "--detach", PIPER_SAMPLE_GENERATOR_COMMIT], check=True)

    PIPER_PYTHON_DIR = f'{PIPER_REPO_DIR}/src/python'
    MA_DIR = f'{PIPER_PYTHON_DIR}/piper_train/vits/monotonic_align'
    MA_IMPORT_DIR = f'{MA_DIR}/monotonic_align'
    MA_BUILD_DIR = f'{MA_DIR}/piper_train/vits/monotonic_align'

    shutil.rmtree(f'{PIPER_PYTHON_DIR}/build', ignore_errors=True)
    shutil.rmtree(MA_IMPORT_DIR, ignore_errors=True)
    shutil.rmtree(f'{MA_DIR}/piper_train', ignore_errors=True)
    os.makedirs(MA_IMPORT_DIR, exist_ok=True)
    os.makedirs(MA_BUILD_DIR, exist_ok=True)
    open(f'{MA_IMPORT_DIR}/__init__.py', 'a').close()
    subprocess.run(f'cd {MA_DIR} && {sys.executable} setup.py build_ext --inplace',
                    shell=True, check=True)
    built = next(iter(glob.glob(f'{MA_BUILD_DIR}/core.*')), None)
    assert built, 'monotonic_align core extension build failed'
    shutil.copy2(built, MA_IMPORT_DIR)

    for path in (PIPER_PYTHON_DIR, PIPER_SAMPLE_GENERATOR_DIR):
        if path not in sys.path:
            sys.path.insert(0, path)

    # The original LibriTTS-R generator is still the default. For other languages,
    # download a standard Piper ONNX voice and its matching JSON configuration.
    from urllib.parse import quote
    os.makedirs('models', exist_ok=True)
    if PIPER_MODEL_TYPE == 'generator':
        MODEL_PATH = 'models/en_US-libritts_r-medium.pt'
        MODEL_CONFIG_PATH = f'{MODEL_PATH}.json'
        if not os.path.exists(MODEL_PATH):
            print('Downloading libritts_r model (~75 MB)...')
            urllib.request.urlretrieve(
                'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt',
                MODEL_PATH)
            urllib.request.urlretrieve(
                'https://huggingface.co/rhasspy/piper-voices/resolve/3ec800d/en/en_US/libritts_r/medium/en_US-libritts_r-medium.onnx.json',
                MODEL_CONFIG_PATH)
    else:
        voice_id = f'{PIPER_LANGUAGE}-{PIPER_VOICE}-{PIPER_QUALITY}'
        MODEL_PATH = f'models/{voice_id}.onnx'
        MODEL_CONFIG_PATH = f'{MODEL_PATH}.json'
        voice_root = f'https://huggingface.co/rhasspy/piper-voices/resolve/main/{PIPER_LANGUAGE.split("_")[0]}/{PIPER_LANGUAGE}/{PIPER_VOICE}/{PIPER_QUALITY}'
        model_url = PIPER_VOICE_URL or f'{voice_root}/{voice_id}.onnx'
        config_url = PIPER_VOICE_CONFIG_URL or f'{voice_root}/{voice_id}.onnx.json'
        if not os.path.exists(MODEL_PATH):
            print(f'Downloading Piper voice {voice_id!r}...')
            urllib.request.urlretrieve(model_url, MODEL_PATH)
        if not os.path.exists(MODEL_CONFIG_PATH):
            urllib.request.urlretrieve(config_url, MODEL_CONFIG_PATH)
        # Piper voice JSON declares the actual phoneme inventory/locale.
        import json
        with open(MODEL_CONFIG_PATH, encoding='utf-8') as f:
            _piper_voice_config = json.load(f)
        print('Piper voice config language:', _piper_voice_config.get('language', {}))
    print('Piper ready')

else:
    print("Skipped (dataset source setting).")


In [ ]:
# === Generated training data (optional) ===
if POSITIVE_SOURCE in ("generated", "both"):
    # === Generate positive samples (US English, optionally UK) ===
    import os, subprocess, sys, shutil
    from pathlib import Path
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'
    PIPER_BATCH = 256  # drop to 128 on T4 if CUDA OOM

    def run_piper(target_word, max_samples, output_dir):
      cmd = [
          sys.executable,
          '-m', 'piper_sample_generator',
          target_word,
          '--phoneme-input',
          '--model', MODEL_PATH,
          '--max-samples', str(max_samples),
          '--batch-size', str(PIPER_BATCH),
          '--noise-scales', '0.5',
          '--noise-scale-ws', '0.6',
          '--output-dir', output_dir,
      ]

      env = os.environ.copy()
      env['PYTHONPATH'] = (
          f'{PIPER_PYTHON_DIR}:'
          f'{PIPER_SAMPLE_GENERATOR_DIR}:'
          + env.get('PYTHONPATH', '')
      )

      subprocess.run(cmd, check=True, env=env)

    os.makedirs('generated_samples', exist_ok=True)

    # Each pass receives a distinct directory; the generator otherwise starts at 0.wav.
    for pass_label, ipa, sample_count in (
        ('us', WAKE_WORD_IPA_US, SAMPLES_US),
        ('uk', WAKE_WORD_IPA_UK, SAMPLES_UK),
    ):
        if not ipa or sample_count <= 0:
            continue
        print(f'Generating {sample_count} {pass_label.upper()} samples for {ipa!r} using {PIPER_LANGUAGE}...')
        temp_dir = f'/content/generated_positive_{pass_label}'
        shutil.rmtree(temp_dir, ignore_errors=True)
        run_piper(ipa, sample_count, temp_dir)
        for f in sorted(Path(temp_dir).glob('*.wav')):
            shutil.move(str(f), f'generated_samples/{pass_label}_{f.name}')
        shutil.rmtree(temp_dir, ignore_errors=True)

    n = sum(1 for f in os.listdir('generated_samples') if f.endswith('.wav'))
    print(f'Total positive samples: {n}')
else:
    print("Skipped (dataset source setting).")


In [ ]:
# === Generated training data (optional) ===
if NEGATIVE_SOURCE in ("generated", "both"):
    import os, subprocess, sys
    from pathlib import Path
    PIPER_SAMPLE_GENERATOR_DIR = '/content/piper-sample-generator'

    os.makedirs('confusable_negatives', exist_ok=True)
    for phrase in CONFUSABLE_PHRASES:
        safe = phrase.replace(' ', '_').replace(',', '').replace("'", '')
        existing = len(list(Path('confusable_negatives').glob(f'{safe}_*.wav')))
        if existing >= SAMPLES_PER_CONFUSABLE:
            print(f'  {phrase!r}: {existing} already, skip')
            continue
        tmp = f'/tmp/confusable_{safe}'
        os.makedirs(tmp, exist_ok=True)
        print(f'  generating {SAMPLES_PER_CONFUSABLE} for {phrase!r}...')
        env = os.environ.copy()
        env['PYTHONPATH'] = (
            f'{PIPER_PYTHON_DIR}:'
            f'{PIPER_SAMPLE_GENERATOR_DIR}:'
            + env.get('PYTHONPATH', '')
        )

        subprocess.run([
            sys.executable,
            '-m', 'piper_sample_generator',
            phrase,
            '--model', MODEL_PATH,
            '--max-samples', str(SAMPLES_PER_CONFUSABLE),
            '--batch-size', '256',
            '--noise-scales', '0.5',
            '--noise-scale-ws', '0.6',
            '--output-dir', tmp,
        ], check=True, env=env)
        # Move + rename with safe prefix
        for f in os.listdir(tmp):
            if f.endswith('.wav'):
                os.rename(f'{tmp}/{f}', f'confusable_negatives/{safe}_{f}')

    n = sum(1 for f in os.listdir('confusable_negatives') if f.endswith('.wav'))
    print(f'Total confusable negatives: {n}')

else:
    print("Skipped (dataset source setting).")


In [ ]:
# === Download negative datasets + augmentation data ===

import os
import zipfile
import subprocess
from huggingface_hub import hf_hub_download

# -------------------------------------------------------
# Negative feature datasets
# -------------------------------------------------------

NEG_ROOT = "/content/negative_datasets"
os.makedirs(NEG_ROOT, exist_ok=True)

negative_archives = {
    "speech": "speech.zip",
    "dinner_party": "dinner_party.zip",
    "no_speech": "no_speech.zip",
    "dinner_party_eval": "dinner_party_eval.zip",
}

if NEGATIVE_SOURCE in ("generated", "both"):
    for target_name, archive_name in negative_archives.items():
        target_dir = os.path.join(NEG_ROOT, target_name)

        if os.path.isdir(target_dir) and os.listdir(target_dir):
            print(f"{target_name}: already present")
            continue

        print(f"Downloading {archive_name}...")

        archive = hf_hub_download(
            repo_id="kahrendt/microwakeword",
            repo_type="dataset",
            filename=archive_name,
            revision="0da95f9",
        )

        os.makedirs(target_dir, exist_ok=True)

        with zipfile.ZipFile(archive, "r") as zf:
            names = zf.namelist()

            # Some archive versions may already contain their own top-level folder.
            top_levels = {
                name.split("/", 1)[0]
                for name in names
                if name and not name.startswith("/")
            }

            if top_levels == {target_name}:
                zf.extractall(NEG_ROOT)
            else:
                zf.extractall(target_dir)

        print(f"{target_name}: ready")

else:
    print("Skipping standard negative training datasets.")

# -------------------------------------------------------
# Room impulse responses
# -------------------------------------------------------

if not os.path.exists("mit_rirs") or not os.listdir("mit_rirs"):
    print("Downloading MIT room impulse responses...")
    os.makedirs("mit_rirs", exist_ok=True)

    subprocess.run(
        [
            "wget", "-q",
            "https://www.openslr.org/resources/28/rirs_noises.zip",
            "-O", "mit_rirs/rirs_noises.zip",
        ],
        check=True,
    )

    subprocess.run(
        ["unzip", "-q", "mit_rirs/rirs_noises.zip", "-d", "mit_rirs"],
        check=True,
    )

print("Negative datasets and RIRs ready")

In [ ]:
# === Prepare FMA background audio ===

import os
import shutil
import subprocess
from pathlib import Path

FMA_SOURCE = Path("/content/fma_source")
FMA_16K = Path("/content/fma_16k")
FMA_REVISION="bfef24fd1e1334e4631b917c98e0dbc3dd653ab2"

# Remove the empty directory left by the failed old download
if FMA_16K.exists():
    shutil.rmtree(FMA_16K)

if FMA_SOURCE.exists():
    shutil.rmtree(FMA_SOURCE)

FMA_SOURCE.mkdir(parents=True)
FMA_16K.mkdir(parents=True)

if BACKGROUND_SOURCE in ('standard', 'both'):
    zip_path = FMA_SOURCE / "fma_xs.zip"

    print("Downloading FMA xsmall...")

    subprocess.run([
        "wget",
        "-q",
        "--show-progress",
        "-O", str(zip_path),
        f"https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/{FMA_REVISION}/fma_xs.zip",
    ], check=True)

    print("Extracting...")
    subprocess.run([
        "unzip",
        "-q",
        str(zip_path),
        "-d", str(FMA_SOURCE),
    ], check=True)

    audio_files = [
        p for p in FMA_SOURCE.rglob("*")
        if p.suffix.lower() in {".mp3", ".wav", ".flac", ".ogg", ".m4a"}
    ]

    print(f"Found {len(audio_files)} source audio files")

    assert audio_files, "FMA archive contained no recognised audio files"

    print("Converting to 16 kHz mono WAV...")

    for i, src in enumerate(audio_files):
        dst = FMA_16K / f"{i:06d}.wav"

        subprocess.run([
            "ffmpeg",
            "-loglevel", "error",
            "-y",
            "-i", str(src),
            "-ar", "16000",
            "-ac", "1",
            "-c:a", "pcm_s16le",
            str(dst),
        ], check=True)

        if (i + 1) % 100 == 0:
            print(f"  converted {i + 1}/{len(audio_files)}")

    wav_files = list(FMA_16K.glob("*.wav"))

    print(f"Ready: {len(wav_files)} files in {FMA_16K}")
    assert wav_files, "fma_16k is still empty"
else:
    print('Skipping FMA background download (uploaded background only).')


In [ ]:
# === Optional positive / negative / background ZIP uploads (Google Colab) ===
# Upload distinct ZIP archives in ONE prompt. Nested folders are supported.
# Filenames must start with positive, negative, or background (e.g. positive_1.zip).
# Multiple ZIP files per dataset are supported and combined. No upload occurs
# when all three source controls select generated/standard data only.
# Audio is converted to mono, 16-kHz, 16-bit PCM WAV before feature extraction.
import io
import subprocess
import zipfile
from pathlib import Path
from google.colab import files

UPLOAD_DESTINATIONS = {
    'positive': Path('/content/uploaded_positive_samples'),
    'negative': Path('/content/uploaded_negative_samples'),
    'background': Path('/content/uploaded_background_audio'),
}
required_uploads = set()
if POSITIVE_SOURCE in ('uploaded', 'both'):
    required_uploads.add('positive')
if NEGATIVE_SOURCE in ('uploaded', 'both'):
    required_uploads.add('negative')
if BACKGROUND_SOURCE in ('uploaded', 'both'):
    required_uploads.add('background')

UPLOAD_COUNTS = {key: 0 for key in UPLOAD_DESTINATIONS}
if required_uploads:
    print('Upload ZIP files together. Required prefixes:', ', '.join(sorted(required_uploads)))
    print('Examples: positive_morgan.zip, negative_confusables.zip, background_room.zip')
    uploaded = files.upload()
    matched = set()
    for filename, data in uploaded.items():
        filename_lower = Path(filename).name.lower()
        if not filename_lower.endswith('.zip'):
            raise ValueError(f'Only ZIP archives are supported: {filename!r}')
        category = next((k for k in required_uploads
                         if filename_lower.startswith(k) and
                         (len(filename_lower) == len(k) + 4 or filename_lower[len(k)] in '_-.')), None)
        if category is None:
            raise ValueError(f'{filename!r} does not match a required dataset prefix: {sorted(required_uploads)}')
        destination = UPLOAD_DESTINATIONS[category]
        destination.mkdir(parents=True, exist_ok=True)
        matched.add(category)
        with zipfile.ZipFile(io.BytesIO(data)) as archive:
            for entry in archive.infolist():
                if entry.is_dir() or Path(entry.filename).suffix.lower() not in {'.wav', '.flac', '.mp3', '.ogg', '.m4a'}:
                    continue
                # Never extract ZIP paths to disk; use sequential safe filenames.
                output = destination / f'{UPLOAD_COUNTS[category]:08d}.wav'
                with archive.open(entry) as input_audio:
                    result = subprocess.run(
                        ['ffmpeg', '-nostdin', '-loglevel', 'error', '-y', '-i', 'pipe:0',
                         '-ar', '16000', '-ac', '1', '-c:a', 'pcm_s16le', str(output)],
                        input=input_audio.read(), capture_output=True)
                if result.returncode != 0:
                    output.unlink(missing_ok=True)
                    raise RuntimeError(f'Could not decode {entry.filename!r}: {result.stderr.decode(errors="replace")}')
                UPLOAD_COUNTS[category] += 1
    if matched != required_uploads:
        raise ValueError(f'Missing ZIP archives for: {sorted(required_uploads - matched)}')
    for category in sorted(required_uploads):
        count = UPLOAD_COUNTS[category]
        if count == 0:
            raise ValueError(f'No supported audio clips in {category} ZIP archive(s)')
        print(f'{category}: {count} files prepared in {UPLOAD_DESTINATIONS[category]}')
        if category != 'background' and count < 10:
            print('WARNING: Small datasets may yield unreliable train/validation/test splits.')
else:
    print('No dataset uploads needed.')


In [ ]:
# === Augmentation + feature extraction ===
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
import os, shutil, traceback
from mmap_ninja.ragged import RaggedMmap

# In uploaded-only mode, use the uploaded clips as the main positive set.
# In both mode, uploaded clips have a separate feature set and configurable weight.
POSITIVE_DIRECTORY = ('uploaded_positive_samples' if POSITIVE_SOURCE == 'uploaded'
                      else 'generated_samples')
clips = Clips(
    input_directory=POSITIVE_DIRECTORY,
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15, 'TanhDistortion': 0.10,
        'PitchShift': 0.15, 'BandStopFilter': 0.10,
        'AddColorNoise': 0.20, 'AddBackgroundNoise': 0.85,
        'Gain': 1.00, 'GainTransition': 0.25, 'RIR': 0.60,
    },
    impulse_paths=['mit_rirs'],
    background_paths=([
        *(['fma_16k'] if BACKGROUND_SOURCE in ('standard', 'both') else []),
        *(['uploaded_background_audio'] if BACKGROUND_SOURCE in ('uploaded', 'both') else []),
    ]),
    background_min_snr_db=-5, background_max_snr_db=20,
    min_jitter_s=0.10, max_jitter_s=0.50,
)

os.makedirs('generated_augmented_features', exist_ok=True)
SPLIT_CONFIG = {
    'training':   {'split_name': 'train',      'repetition': 3, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
}

for split, cfg in SPLIT_CONFIG.items():
    out = f'generated_augmented_features/{split}'
    mmap = f'{out}/wakeword_mmap'
    if os.path.exists(mmap) and list(os.scandir(mmap)):
        print(f'{split}: cached, skipping')
        continue
    if os.path.exists(mmap):
        shutil.rmtree(mmap)
    os.makedirs(out, exist_ok=True)
    print(f'Generating {split} (rep={cfg["repetition"]}, slide={cfg["slide_frames"]})...')
    try:
        sg = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    except Exception:
        traceback.print_exc()
        if os.path.exists(mmap): shutil.rmtree(mmap)
        raise
print('Positive features ready')

# Confusable features (only if confusable_negatives/ exists)
if NEGATIVE_SOURCE in ('generated', 'both') and os.path.exists('confusable_negatives') and os.listdir('confusable_negatives'):
    print('Generating confusable features...')
    confusable_clips = Clips(
        input_directory='confusable_negatives', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('confusable_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'confusable_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=confusable_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Confusable features ready')

# Real recording features (only if real_recordings/ exists)
if POSITIVE_SOURCE in ('generated', 'both') and os.path.exists('real_recordings') and os.listdir('real_recordings'):
    print('Generating real-recording features...')
    real_clips = Clips(
        input_directory='real_recordings', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('real_recording_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'real_recording_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=real_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Real-recording features ready')

# Uploaded data features: one extra positive set in "both" mode, and
# one negative set in either upload-enabled mode.
def make_uploaded_features(input_directory, feature_directory):
    uploaded_clips = Clips(
        input_directory=input_directory, file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    for split, cfg in SPLIT_CONFIG.items():
        out = f'{feature_directory}/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap):
            shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(
            clips=uploaded_clips, augmenter=augmenter,
            slide_frames=cfg['slide_frames'], step_ms=10,
        )
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print(f'{feature_directory}: ready')

if POSITIVE_SOURCE == 'both':
    make_uploaded_features('uploaded_positive_samples', 'uploaded_positive_features')
if NEGATIVE_SOURCE in ('uploaded', 'both'):
    make_uploaded_features('uploaded_negative_samples', 'uploaded_negative_features')


In [ ]:
# === Listen to generated samples ONLY (never play uploaded datasets) ===
# The generated positive and confusable-negative sample previews are independent
# of the uploaded datasets. Re-run this cell to hear fresh random selections.
from pathlib import Path
from IPython.display import Audio, display
import random

GENERATED_PREVIEW_COUNT = 10
preview_sets = [
    ('Generated positive samples', 'generated_samples', POSITIVE_SOURCE),
    ('Generated confusable negatives', 'confusable_negatives', NEGATIVE_SOURCE),
]
for label, folder, source_mode in preview_sets:
    if source_mode not in ('generated', 'both'):
        print(f'{label}: skipped (source={source_mode})')
        continue
    preview_files = list(Path('/content', folder).rglob('*.wav'))
    print(f'{label}: {len(preview_files)} available')
    for f in random.sample(preview_files, min(GENERATED_PREVIEW_COUNT, len(preview_files))):
        display(Audio(str(f)))


In [ ]:
# === Training config YAML ===
import yaml, os
from pathlib import Path

SKIP_CONFUSABLES = not Path('confusable_features/training/wakeword_mmap').exists()
SKIP_REAL = not Path('real_recording_features/training/wakeword_mmap').exists()

config = {
    'window_step_ms': 10,
    'train_dir': f'trained_models/{OUTPUT_NAME}',
    'features': [],
    'training_steps': [25000, 20000],
    'positive_class_weight': [2, 2],
    'negative_class_weight': [40, 50],
    'learning_rates': [0.001, 0.0001],
    'batch_size': 256,
    'time_mask_max_size': [5, 5], 'time_mask_count': [1, 1],
    'freq_mask_max_size': [3, 3], 'freq_mask_count': [1, 1],
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.4,
    'minimization_metric': 'ambient_false_positives_per_hour',
    'maximization_metric': 'average_viable_recall',
}

# The base positive dataset is generated, uploaded, or a combination.
config['features'].append(dict(
    features_dir='generated_augmented_features', sampling_weight=8.0,
    penalty_weight=2.0, truth=True, truncation_strategy='truncate_start', type='mmap'))

if NEGATIVE_SOURCE in ('generated', 'both'):
    config['features'].extend([
        dict(features_dir='negative_datasets/speech', sampling_weight=10.0,
             penalty_weight=2.5, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party', sampling_weight=15.0,
             penalty_weight=3.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/no_speech', sampling_weight=5.0,
             penalty_weight=1.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party_eval', sampling_weight=0.0,
             penalty_weight=1.0, truth=False, truncation_strategy='split', type='mmap'),
    ])
if NEGATIVE_SOURCE in ('uploaded', 'both'):
    config['features'].append(dict(
        features_dir='uploaded_negative_features', sampling_weight=10.0,
        penalty_weight=3.0, truth=False, truncation_strategy='random', type='mmap'))
if POSITIVE_SOURCE == 'both':
    config['features'].append(dict(
        features_dir='uploaded_positive_features', sampling_weight=8.0,
        penalty_weight=2.0, truth=True, truncation_strategy='truncate_start', type='mmap'))

if NEGATIVE_SOURCE in ('generated', 'both') and not SKIP_CONFUSABLES:
    config['features'].append(dict(features_dir='confusable_features', sampling_weight=8.0,
                                    penalty_weight=5.0, truth=False,
                                    truncation_strategy='random', type='mmap'))
if POSITIVE_SOURCE in ('generated', 'both') and not SKIP_REAL:
    config['features'].append(dict(features_dir='real_recording_features', sampling_weight=8.0,
                                    penalty_weight=2.0, truth=True,
                                    truncation_strategy='truncate_start', type='mmap'))

os.makedirs(f'trained_models/{OUTPUT_NAME}', exist_ok=True)
with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)
print('training_parameters.yaml ready')
print(f'  Feature sets: {len(config["features"])}, total steps: {sum(config["training_steps"])}')


In [ ]:
# === Train the model ===
# subprocess + PYTHONPATH so it can find microwakeword (sys.path doesn't propagate)
import os, sys, subprocess, shutil
shutil.rmtree(f'trained_models/{OUTPUT_NAME}', ignore_errors=True)

env = os.environ.copy()
env['PYTHONPATH'] = '/content/microWakeWord:' + env.get('PYTHONPATH', '')
env['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'

cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config', 'training_parameters.yaml',
    '--train', '1', '--restore_checkpoint', '0',
    '--test_tflite_streaming_quantized', '1',
    '--use_weights', 'best_weights',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1, 1, 1, 1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '3',
]
print('Running:', ' '.join(cmd)); print()
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(); print('Exit code:', proc.returncode)
assert proc.returncode == 0, 'training failed - see output above'


In [ ]:
# === Export + browser download ===
import os
import json
import shutil
from google.colab import files

tflite_src = (
    f"trained_models/{OUTPUT_NAME}/"
    "tflite_stream_state_internal_quant/"
    "stream_state_internal_quant.tflite"
)

assert os.path.exists(tflite_src), f"No model at {tflite_src}"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TFLITE_NAME = f"{OUTPUT_NAME}.tflite"
JSON_NAME = f"{OUTPUT_NAME}.json"

OUT_TFLITE = os.path.join(OUTPUT_DIR, TFLITE_NAME)
OUT_JSON = os.path.join(OUTPUT_DIR, JSON_NAME)

shutil.copy2(tflite_src, OUT_TFLITE)
print(f"wrote {OUT_TFLITE} ({os.path.getsize(OUT_TFLITE)/1024:.1f} KB)")

manifest = {
    "type": "micro",
    "wake_word": WAKE_WORD,
    "author": AUTHOR,
    "website": AUTHOR_WEBSITE,
    "model": TFLITE_NAME,
    "trained_languages": TRAINED_LANGUAGES,
    "version": 2,
    "micro": {
        "probability_cutoff": PROBABILITY_CUTOFF,
        "feature_step_size": 10,
        "sliding_window_size": SLIDING_WINDOW_SIZE,
        "tensor_arena_size": TENSOR_ARENA_SIZE,
        "minimum_esphome_version": "2024.7.0",
    },
}

with open(OUT_JSON, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"wrote {OUT_JSON}")
print(json.dumps(manifest, indent=2))

# Package both files into one download
ZIP_BASE = f"/content/{OUTPUT_NAME}_model"
ZIP_FILE = shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir=OUTPUT_DIR,
)

print(f"Downloading {ZIP_FILE}")
files.download(ZIP_FILE)

## Deploying to ESPHome devices

Drop the `.tflite` + `.json` next to your ESPHome YAML (e.g. in `/config/esphome/wakewords/<output_name>/`).

In your device YAML, replace your existing wake word:

```yaml
micro_wake_word:
  models:
    - model: wakewords/<output_name>/<output_name>.json
  on_wake_word_detected:
    - voice_assistant.start:
```

Then `esphome run <device>.yaml` (USB or OTA).

## Manifest tuning (likely needed)

The defaults work for **Hey Harold** specifically. Your model's confidence
distribution will differ. Iterate:

| Symptom | Knob |
|---|---|
| Doesn't fire on the wake word | Lower `probability_cutoff` (try 0.7, 0.6) |
| Fires on too many things | Raise `probability_cutoff` (try 0.92, 0.95) |
| LED fires but no STT response | `sliding_window_size: 5` (faster fire); also check Echo speaker mute |
| `Failed to allocate tensors` log | Raise `tensor_arena_size` (try 50000, 80000) |

## Known gotchas
- Editable install (`pip install -e ./microWakeWord`) requires kernel restart — broken for Run All
- `train.py` upstream calls `.numpy()` on numpy arrays under newer TF (this notebook patches it)
- T4 GPU OOMs during validation — use A100 + High-RAM
- Manifest path mismatch: training writes to `trained_models/<output_name>/`, manifest must match
- `voice_assistant` has no audio lookback; if MWW fires AFTER user speaks, STT gets silence
